## Эксперименты на этапе генерации (проверяем разные модели)

In [1]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login
from tqdm import tqdm
import os
from datetime import datetime
import warnings
import time
import numpy as np
import gc


warnings.filterwarnings('ignore')

HF_TOKEN = ""  

if HF_TOKEN and HF_TOKEN != "hf_your_token_here":
    login(token=HF_TOKEN)
    print("✅ Авторизация на Hugging Face выполнена")
else:
    try:
        from huggingface_hub import whoami
        whoami()
        print("✅ Уже авторизованы")
    except:
        print("⚠️ Требуется авторизация. Запустите: huggingface-cli login")
        login()
        
class RAGAnswerGenerator:

    def __init__(self, model_name, device=None, max_context_tokens=4096, use_quantization=False):
        """
        Инициализация генератора ответов на основе RAG с оптимизациями
        """
        self.model_name = model_name
        self.max_context_tokens = max_context_tokens
        self.use_quantization = use_quantization
    
        if device is None:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"
        else:
            self.device = device
    
        if torch.cuda.is_available() and self.device == "cuda":
            os.environ["CUDA_VISIBLE_DEVICES"] = "0"
            torch.cuda.set_device(0)
            self.device = "cuda:0"
            print(f"Используется GPU: {torch.cuda.get_device_name(0)}")
    
        print(f"Загрузка модели {model_name} на {self.device}...")
  
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()
    
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True,
            padding_side="left",
            truncation_side="left",
            use_fast=True,
            token=HF_TOKEN
        )
    
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
    
        model_kwargs = {
            "trust_remote_code": True,
            "low_cpu_mem_usage": True,
            "use_cache": True,
            "torch_dtype": torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            "token": HF_TOKEN
        }
        
        if use_quantization and torch.cuda.is_available():
            try:
                from transformers import BitsAndBytesConfig
                quantization_config = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_compute_dtype=torch.bfloat16,
                    bnb_4bit_use_double_quant=True,
                    bnb_4bit_quant_type="nf4"
                )
                model_kwargs["quantization_config"] = quantization_config
                print("🔧 Используется 4-битная квантизация")
            except ImportError:
                print("⚠️ BitsAndBytes не установлен, пропускаем квантизацию")
        
        model_lower = model_name.lower()
    
        if torch.cuda.is_available() and not use_quantization:
            # Используем конкретное устройство вместо "auto" для избежания multi-GPU проблем
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                device_map=self.device,  # Явно указываем устройство
                **model_kwargs
            )
        else:
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                device_map="auto" if torch.cuda.is_available() else None,
                **model_kwargs
            )
    
        if "qwen3" in model_lower and torch.cuda.is_available():
            self.model = self.model.to(self.device)
    
        self.model.eval()
        
        if torch.cuda.is_available():
            torch.backends.cudnn.benchmark = True
            
            self.model.generation_config.cache_implementation = "dynamic"
            
            if "qwen3" in model_lower:
                self.model.generation_config.use_cache = True
    
        print(f"Модель загружена. Используется {self.model.device}")
        
        if torch.cuda.is_available():
            print(f"GPU Memory allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
            print(f"GPU Memory cached: {torch.cuda.memory_reserved()/1024**3:.2f} GB")
        
        self.prompt_cache = {}

    def load_data(self, questions_file):
        """Загрузка данных из JSON файла с вопросами и склеенными чанками"""
        print(f"Загрузка вопросов из {questions_file}...")
        with open(questions_file, 'r', encoding='utf-8') as f:
            questions_data = json.load(f)
        
        if isinstance(questions_data, list):
            questions = questions_data
        elif isinstance(questions_data, dict):
            if 'questions' in questions_data:
                questions = questions_data['questions']
            else:
                questions = list(questions_data.values())
        else:
            raise ValueError("Неизвестный формат файла с вопросами")
        
        for i, item in enumerate(questions):
            if isinstance(item, dict):
                if 'question' not in item:
                    print(f"⚠️ Вопрос {i} не имеет поля 'question'")
                if 'texts' not in item:
                    print(f"⚠️ Вопрос {i} не имеет поля 'texts'")
                    item['texts'] = []
        
        print(f"Загружено {len(questions)} вопросов")

        total_texts = sum(len(item.get('texts', [])) for item in questions)
        print(f"Всего текстов (чанков): {total_texts}")
        print(f"Среднее количество текстов на вопрос: {total_texts/len(questions):.1f}")
        
        return questions

    def build_prompt_enhanced(self, question, texts, max_texts=3, include_context=True, style="detailed"):
        """
        Улучшенное создание промпта с развернутыми ответами
        
        Args:
            question: вопрос
            texts: список текстов
            max_texts: максимальное количество текстов для использования
            include_context: включать ли контекст в промпт
            style: "detailed" (развернутый), "concise" (краткий), "comprehensive" (исчерпывающий)
        """
        cache_key = f"{question}_{hash(''.join(texts[:max_texts]))}_{include_context}_{style}"
        
        if cache_key in self.prompt_cache:
            return self.prompt_cache[cache_key]
        
        if not texts:
            prompt = f"""Question: {question}

Please provide a comprehensive answer based on your knowledge. Be specific and include relevant details, examples, and explanations where appropriate.

Answer:"""
        elif include_context and texts:
            selected_texts = texts[:max_texts]

            context_parts = []
            for i, text in enumerate(selected_texts, 1):
                truncated_text = text[:800] if len(text) > 800 else text
                context_parts.append(f"[Source {i}]\n{truncated_text}")
            
            context = "\n\n".join(context_parts)
            
            if style == "detailed":
                prompt = f"""You are an expert IT analyst. 
                Based on the provided sources, generate a detailed, informative answer to the question.

**Guidelines for your response:**
- Provide a comprehensive answer with specific details from the sources
- Include key facts, numbers, dates, and technical details where available
- Organize information logically (e.g., chronologically, by importance, or by category)
- Use examples from the sources to support your points
- If multiple perspectives exist, mention them
- Keep the answer focused on IT and technology aspects
- Aim for 3-5 paragraphs of substantive content

**Sources:**
{context}

**Question:**
{question}

**Detailed Answer:**"""
            
            elif style == "comprehensive":
                prompt = f"""You are a senior technology journalist. Create a thorough, well-structured response based exclusively on the sources below.

**Response Structure:**
1. **Executive Summary** (1-2 sentences)
2. **Key Details** (main facts, numbers, announcements)
3. **Technical Analysis** (deeper dive into technical aspects)
4. **Implications** (what this means for the industry)
5. **Sources Reference** (mention which source provided which information)

**Sources:**
{context}

**Question:**
{question}

**Comprehensive Answer:**"""
            
            else:
                prompt = f"""Answer the question using ONLY the information from the sources.

**STRICT RULES:**
1. Use ONLY facts explicitly stated in the sources.
2. Do NOT add assumptions, general knowledge, or interpretations.
3. If the answer is not clearly supported by the sources, say: "The sources do not provide enough information."
4. Copy all numbers, names, and specific details EXACTLY as written.
5. Answer ONLY the question — do not include unrelated information.

**STYLE:**
- Be concise and precise.
- Do not repeat the question.
- Do not summarize the entire context.
- Prefer short factual statements over long explanations.

**PROCESS (follow internally):**
- Identify the parts of the sources directly relevant to the question.
- Ignore irrelevant information.
- Extract and combine only the necessary facts.

**Sources:**
{context}

**Question:**
{question}

**Answer:**"""
        
        else:
            prompt = f"""Question: {question}

Please provide a clear and informative answer.

Answer:"""
        
        if len(self.prompt_cache) < 100:
            self.prompt_cache[cache_key] = prompt
            
        return prompt

    def generate_answer_optimized(self, prompt, max_new_tokens=256, temperature=0.5, top_p=0.9, repetition_penalty=1.05):
        """
        Оптимизированная генерация ответа с улучшенными параметрами
        """
        inputs = self.tokenizer(
            prompt, 
            return_tensors="pt", 
            truncation=True, 
            max_length=self.max_context_tokens
        )

        if torch.cuda.is_available():
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.no_grad():
            try:
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    temperature=temperature,
                    do_sample=temperature > 0,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                    top_p=top_p if temperature > 0 else 1.0,
                    repetition_penalty=repetition_penalty,
                    use_cache=True,
                    num_beams=1,
                    early_stopping=True,
                    no_repeat_ngram_size=3
                )
            except Exception as e:
                print(f"Warning: {e}, retrying without cache")
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    temperature=temperature,
                    do_sample=temperature > 0,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                    top_p=top_p,
                    repetition_penalty=repetition_penalty,
                    use_cache=False,
                    num_beams=1,
                    early_stopping=True
                )

        generated_text = self.tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:], 
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False 
        )
        
        answer = generated_text.strip()
        
        if answer.startswith("Answer:"):
            answer = answer.replace("Answer:", "").strip()
        if answer.startswith("Detailed Answer:"):
            answer = answer.replace("Detailed Answer:", "").strip()
        
        answer = '\n'.join(line.strip() for line in answer.split('\n') if line.strip())
        
        return answer

    def process_questions_optimized(
        self, 
        questions, 
        max_texts=3, 
        output_file=None,
        max_new_tokens=256,
        temperature=0.5,
        include_context=True,
        response_style="detailed",
        batch_size=4,
        save_intermediate=True
    ):
        """
        Оптимизированная обработка всех вопросов с поддержкой batch processing
        
        Args:
            questions: список вопросов
            max_texts: максимальное количество текстов
            output_file: файл для сохранения результатов
            max_new_tokens: максимальное количество новых токенов
            temperature: температура генерации
            include_context: включать контекст
            response_style: "detailed", "concise", или "comprehensive"
            batch_size: размер батча для параллельной обработки
            save_intermediate: сохранять промежуточные результаты
        """
        results = []
        
        print(f"\n{'='*60}")
        print(f"НАЧАЛО ГЕНЕРАЦИИ (ОПТИМИЗИРОВАННАЯ ВЕРСИЯ)")
        print(f"{'='*60}")
        print(f"Всего вопросов: {len(questions)}")
        print(f"Максимум текстов на вопрос: {max_texts}")
        print(f"Включать контекст: {include_context}")
        print(f"Стиль ответов: {response_style}")
        print(f"Max new tokens: {max_new_tokens}")
        print(f"Temperature: {temperature}")
        print(f"Batch size: {batch_size}")
        print(f"{'='*60}\n")

        start_time = time.time()
        successful = 0
        failed = 0
        total_tokens = 0

        for batch_start in tqdm(range(0, len(questions), batch_size), desc="Генерация ответов"):
            batch_end = min(batch_start + batch_size, len(questions))
            batch_questions = questions[batch_start:batch_end]
            
            batch_prompts = []
            batch_metadata = []
            
            for idx, question_item in enumerate(batch_questions):
                try:
                    if isinstance(question_item, dict):
                        question_text = question_item.get('question', '')
                        texts = question_item.get('texts', [])
                        if isinstance(texts, str):
                            texts = [texts]
                    else:
                        question_text = str(question_item)
                        texts = []
                    
                    if not question_text:
                        answer = "No question provided."
                        batch_prompts.append(None)
                        batch_metadata.append((question_text, texts, idx))
                    elif not texts:
                        prompt = self.build_prompt_enhanced(question_text, [], include_context=False, style=response_style)
                        batch_prompts.append(prompt)
                        batch_metadata.append((question_text, texts, idx))
                    else:
                        prompt = self.build_prompt_enhanced(question_text, texts, max_texts, include_context, style=response_style)
                        batch_prompts.append(prompt)
                        batch_metadata.append((question_text, texts, idx))
                        
                except Exception as e:
                    print(f"\n❌ Ошибка подготовки вопроса {batch_start + idx}: {e}")
                    failed += 1
                    results.append({
                        'index': batch_start + idx,
                        'question': question_text if 'question_text' in locals() else 'Unknown',
                        'error': str(e),
                        'timestamp': datetime.now().isoformat()
                    })

            batch_answers = []
            for prompt in batch_prompts:
                if prompt is None:
                    batch_answers.append("No question provided.")
                else:
                    try:
                        answer = self.generate_answer_optimized(
                            prompt, 
                            max_new_tokens=max_new_tokens,
                            temperature=temperature
                        )
                        batch_answers.append(answer)
                    except Exception as e:
                        print(f"\n❌ Ошибка генерации: {e}")
                        batch_answers.append(f"Error: {str(e)}")
                        failed += 1

            for (question_text, texts, relative_idx), answer in zip(batch_metadata, batch_answers):
                if not question_text:
                    continue
                    
                token_count = len(answer.split())
                total_tokens += token_count
                successful += 1
                
                result = {
                    'index': batch_start + relative_idx,
                    'question': question_text,
                    'texts_used': texts[:max_texts] if texts else [],
                    'generated_answer': answer,
                    'answer_length': len(answer),
                    'token_count': token_count,
                    'model_used': self.model_name,
                    'timestamp': datetime.now().isoformat(),
                    'include_context': include_context,
                    'max_new_tokens': max_new_tokens,
                    'temperature': temperature,
                    'response_style': response_style
                }
                
                results.append(result)
            
            if save_intermediate and output_file and (batch_end) % (batch_size * 5) == 0:
                self.save_results(results, output_file)
                elapsed = time.time() - start_time
                rate = (batch_end) / elapsed
                print(f"\n⏱️  Прогресс: {batch_end}/{len(questions)}")
                print(f"⚡ Скорость: {rate:.2f} вопросов/сек")
                print(f"📊 Успешно: {successful}, Ошибок: {failed}")
        
        elapsed = time.time() - start_time
        
        print(f"\n{'='*60}")
        print(f"ГЕНЕРАЦИЯ ЗАВЕРШЕНА")
        print(f"{'='*60}")
        print(f"✅ Успешно: {successful}")
        print(f"❌ Ошибок: {failed}")
        print(f"⏱️  Общее время: {elapsed:.1f} сек")
        print(f"⚡ Средняя скорость: {len(results)/elapsed:.2f} вопросов/сек")
        print(f"🔤 Всего токенов: {total_tokens}")
        if successful > 0:
            print(f"📏 Средняя длина ответа: {np.mean([len(r.get('generated_answer', '')) for r in results if 'error' not in r]):.0f} символов")
            print(f"📊 Среднее количество токенов: {total_tokens/successful:.1f} на вопрос")
        print(f"{'='*60}")
        
        return results
    
    def save_results(self, results, output_file):
        """Сохранение результатов в JSON файл"""
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"💾 Результаты сохранены в {output_file}")
    
    def save_as_csv(self, results, output_file):
        """Сохранение результатов в CSV для удобного просмотра"""
        import pandas as pd
        
        df_data = []
        for r in results:
            if 'error' not in r:
                df_data.append({
                    'index': r['index'],
                    'question': r['question'][:200] + '...' if len(r['question']) > 200 else r['question'],
                    'answer': r['generated_answer'],
                    'answer_length': r['answer_length'],
                    'token_count': r['token_count'],
                    'texts_count': len(r.get('texts_used', []))
                })
        
        df = pd.DataFrame(df_data)
        df.to_csv(output_file, index=False, encoding='utf-8-sig')
        print(f"📊 CSV сохранен в {output_file}")
    
    def prepare_for_llm_judge_simple(self, results_file, judge_output_file=None):
        """Подготовка данных для оценки LLM-as-a-Judge"""
        with open(results_file, 'r', encoding='utf-8') as f:
            results = json.load(f)
        
        judge_data = []
        
        for result in results:
            if 'error' not in result:
                judge_item = {
                    'question': result['question'],
                    'generated_answer': result['generated_answer'],
                    'contexts': result.get('texts_used', []),
                    'metadata': {
                        'model_used': result.get('model_used', ''),
                        'answer_length': result.get('answer_length', 0),
                        'include_context': result.get('include_context', True),
                        'response_style': result.get('response_style', 'detailed')
                    }
                }
                judge_data.append(judge_item)
        
        if judge_output_file:
            with open(judge_output_file, 'w', encoding='utf-8') as f:
                json.dump(judge_data, f, ensure_ascii=False, indent=2)
            print(f"📋 Данные для LLM-оценки сохранены в {judge_output_file}")
        
        return judge_data


def main():
    # ============================================
    # НАСТРОЙКИ (ИЗМЕНИТЕ ПОД ВАШИ ДАННЫЕ)
    # ============================================
    
    # Пути к файлам
    QUESTIONS_FILE = "/kaggle/input/crunch/retrieved_chunks_for_generator.json"
    OUTPUT_FILE = "generated_answers_detailed.json"
    JUDGE_OUTPUT_FILE = "judge_evaluation_data_detailed.json"
    CSV_OUTPUT_FILE = "generated_answers_detailed.csv"
    
    # ============================================
    # ДОСТУПНЫЕ МОДЕЛИ (раскомментируйте нужную)
    # ============================================
    
    # Альтернативные модели для IT новостей:
    #MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"  
    #MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  
    #MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"  
    #MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"  
    #MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"
    
    # Более мощные модели (требуют больше памяти):
    # MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"  
    MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"  
    # MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct" 
    
    
    # Режимы генерации (раскомментируйте нужный):
    
    # 1. РАЗВЕРНУТЫЙ РЕЖИМ (для детальных ответов)
    MAX_TEXTS = 3  # Используем больше контекста
    MAX_NEW_TOKENS = 512  # Больше токенов для развернутых ответов
    TEMPERATURE = 0.5  # Немного выше для разнообразия
    RESPONSE_STYLE = "concise"  # detailed, comprehensive, или concise
    
    
    INCLUDE_CONTEXT = True
    SAVE_INTERMEDIATE = True
    BATCH_SIZE = 4 
    USE_QUANTIZATION = False 
    
    print("\n" + "="*60)
    print("🚀 RAG ГЕНЕРАТОР ОТВЕТОВ (ОПТИМИЗИРОВАННАЯ ВЕРСИЯ)")
    print("="*60)
    print(f"📁 Файл с вопросами: {QUESTIONS_FILE}")
    print(f"🤖 Модель: {MODEL_NAME}")
    print(f"⚙️  Параметры:")
    print(f"   - MAX_TEXTS: {MAX_TEXTS}")
    print(f"   - MAX_NEW_TOKENS: {MAX_NEW_TOKENS}")
    print(f"   - TEMPERATURE: {TEMPERATURE}")
    print(f"   - RESPONSE_STYLE: {RESPONSE_STYLE}")
    print(f"   - BATCH_SIZE: {BATCH_SIZE}")
    print(f"   - QUANTIZATION: {USE_QUANTIZATION}")
    print("="*60 + "\n")
    
    
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    generator = RAGAnswerGenerator(
        MODEL_NAME, 
        device=device,
        max_context_tokens=512,
        use_quantization=USE_QUANTIZATION
    )
    
    questions = generator.load_data(QUESTIONS_FILE)
    
    results = generator.process_questions_optimized(
        questions,
        max_texts=MAX_TEXTS,
        output_file=OUTPUT_FILE,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        include_context=INCLUDE_CONTEXT,
        response_style=RESPONSE_STYLE,
        batch_size=BATCH_SIZE,
        save_intermediate=SAVE_INTERMEDIATE
    )
    
    generator.save_results(results, OUTPUT_FILE)
    generator.save_as_csv(results, CSV_OUTPUT_FILE)

    generator.prepare_for_llm_judge_simple(OUTPUT_FILE, JUDGE_OUTPUT_FILE)
    
    print("\n" + "="*60)
    print("✅ ГЕНЕРАЦИЯ УСПЕШНО ЗАВЕРШЕНА!")
    print("="*60)
    print(f"📁 Результаты сохранены в: {OUTPUT_FILE}")
    print(f"📊 CSV файл: {CSV_OUTPUT_FILE}")
    print(f"📋 Данные для LLM-оценки: {JUDGE_OUTPUT_FILE}")
    print("="*60)


if __name__ == "__main__":
    main()

✅ Авторизация на Hugging Face выполнена

🚀 RAG ГЕНЕРАТОР ОТВЕТОВ (ОПТИМИЗИРОВАННАЯ ВЕРСИЯ)
📁 Файл с вопросами: /kaggle/input/crunch/retrieved_chunks_for_generator.json
🤖 Модель: Qwen/Qwen2.5-7B-Instruct
⚙️  Параметры:
   - MAX_TEXTS: 3
   - MAX_NEW_TOKENS: 512
   - TEMPERATURE: 0.5
   - RESPONSE_STYLE: concise
   - BATCH_SIZE: 4
   - QUANTIZATION: False

Используется GPU: Tesla T4
Загрузка модели Qwen/Qwen2.5-7B-Instruct на cuda:0...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

💾 Результаты сохранены в generated_answers_detailed.json

⏱️  Прогресс: 40/80
⚡ Скорость: 0.01 вопросов/сек
📊 Успешно: 40, Ошибок: 0


Генерация ответов:  75%|███████▌  | 15/20 [1:11:29<23:45, 285.00s/it]

💾 Результаты сохранены в generated_answers_detailed.json

⏱️  Прогресс: 60/80
⚡ Скорость: 0.01 вопросов/сек
📊 Успешно: 60, Ошибок: 0


Генерация ответов: 100%|██████████| 20/20 [1:35:28<00:00, 286.41s/it]

💾 Результаты сохранены в generated_answers_detailed.json

⏱️  Прогресс: 80/80
⚡ Скорость: 0.01 вопросов/сек
📊 Успешно: 80, Ошибок: 0

ГЕНЕРАЦИЯ ЗАВЕРШЕНА
✅ Успешно: 80
❌ Ошибок: 0
⏱️  Общее время: 5728.2 сек
⚡ Средняя скорость: 0.01 вопросов/сек
🔤 Всего токенов: 29589
📏 Средняя длина ответа: 2547 символов
📊 Среднее количество токенов: 369.9 на вопрос
💾 Результаты сохранены в generated_answers_detailed.json
📊 CSV сохранен в generated_answers_detailed.csv
📋 Данные для LLM-оценки сохранены в judge_evaluation_data_detailed.json

✅ ГЕНЕРАЦИЯ УСПЕШНО ЗАВЕРШЕНА!
📁 Результаты сохранены в: generated_answers_detailed.json
📊 CSV файл: generated_answers_detailed.csv
📋 Данные для LLM-оценки: judge_evaluation_data_detailed.json


### Оценка сгенерированных ответов подходом LLM-as-a-Judge

In [2]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
import time
import numpy as np
from typing import List, Dict, Optional
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')


class LLMJudge:
    """
    Оценка качества ответов с использованием LLM-as-a-Judge
    """
    
    def __init__(self, model_name, device=None, use_quantization=False):
        """
        Инициализация судьи-модели
        
        Args:
            model_name: имя модели для оценки (рекомендуется Qwen2.5-7B или Phi-3.5)
            device: устройство (cuda/cpu)
            use_quantization: использовать 4-битную квантизацию
        """
        self.model_name = model_name
        
        if device is None:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"
        else:
            self.device = device
        
        print(f"Загрузка модели-судьи {model_name} на {self.device}...")
        
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True,
            padding_side="left",
            use_fast=True
        )
        
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        model_kwargs = {
            "trust_remote_code": True,
            "low_cpu_mem_usage": True,
            "torch_dtype": torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        }
        
        if use_quantization and torch.cuda.is_available():
            try:
                from transformers import BitsAndBytesConfig
                quantization_config = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_compute_dtype=torch.bfloat16,
                    bnb_4bit_use_double_quant=True,
                )
                model_kwargs["quantization_config"] = quantization_config
                print("🔧 Используется 4-битная квантизация")
            except ImportError:
                print("⚠️ BitsAndBytes не установлен")
        
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto" if torch.cuda.is_available() else None,
            **model_kwargs
        )
        
        self.model.eval()
        
        if torch.cuda.is_available():
            torch.backends.cudnn.benchmark = True
        
        print(f"Модель-судья загружена")
        
    def build_judge_prompt(self, question: str, answer: str, contexts: List[str], 
                          criteria: str = "comprehensive") -> str:
        """
        Создание промпта для оценки ответа
        
        Args:
            question: вопрос пользователя
            answer: сгенерированный ответ
            contexts: контексты (чанки), использованные для генерации
            criteria: критерии оценки (comprehensive/rag/summary)
        """
        
        # Ограничиваем контексты для длины
        truncated_contexts = [ctx[:500] + "..." if len(ctx) > 500 else ctx for ctx in contexts[:3]]
        context_text = "\n\n".join([f"[Source {i+1}]\n{ctx}" for i, ctx in enumerate(truncated_contexts)])
        
        if criteria == "rag":
            prompt = f"""You are an expert evaluator for RAG (Retrieval-Augmented Generation) systems. Evaluate the quality of the generated answer based on the provided question and source contexts.

**Question:**
{question}

**Source Contexts:**
{context_text}

**Generated Answer:**
{answer}

**Evaluation Criteria (rate each from 1-5, where 5 is best):**

1. **RELEVANCE (1-5)** - How well does the answer address the question? Does it answer what was asked?
2. **ACCURACY (1-5)** - Is the answer factually consistent with the source contexts? Any hallucinations?
3. **COMPLETENESS (1-5)** - Does the answer cover all important information from the contexts?
4. **CONCISENESS (1-5)** - Is the answer appropriately concise without missing key details?
5. **GROUNDEDNESS (1-5)** - Is the answer properly grounded in the provided sources?

**Provide your evaluation in the following JSON format:**
```json
{{
    "relevance": <score>,
    "accuracy": <score>,
    "completeness": <score>,
    "conciseness": <score>,
    "groundedness": <score>,
    "total_score": <sum of all scores>,
    "strengths": "<brief positive aspects>",
    "weaknesses": "<brief areas for improvement>",
    "verdict": "excellent|good|fair|poor"
}}
```

"""
        else:  # summary
            prompt = f"""Evaluate this summary for quality:

**Question:** {question}

**Answer:** {answer}

Rate from 1-5:
- Relevance: does it answer the question?
- Accuracy: is it correct based on context?
- Completeness: does it cover key points?
- Clarity: is it well-written?

Output JSON:
```json
{{
    "relevance": <score>,
    "accuracy": <score>,
    "completeness": <score>,
    "clarity": <score>,
    "total_score": <sum>,
    "verdict": "excellent|good|fair|poor"
}}
```"""

        return prompt
    
    def parse_judge_response(self, response: str) -> Dict:
        """
        Парсинг ответа судьи в структурированный формат
        """
        try:
            import re
            json_match = re.search(r'```json\s*(\{.*?\})\s*```', response, re.DOTALL)
            if json_match:
                json_str = json_match.group(1)
            else:
                json_match = re.search(r'\{.*?\}', response, re.DOTALL)
                if json_match:
                    json_str = json_match.group(0)
                else:
                    raise ValueError("JSON not found")
            
            result = json.loads(json_str)
            
            if 'total_score' not in result:
                score_keys = [k for k in result.keys() if k not in ['strengths', 'weaknesses', 'verdict']]
                result['total_score'] = sum(result.get(k, 0) for k in score_keys)
            
            return result
            
        except Exception as e:
            print(f"⚠️ Ошибка парсинга: {e}")
            return {
                "error": str(e),
                "raw_response": response[:200],
                "total_score": 0,
                "verdict": "parse_error"
            }
    
    def evaluate_single(self, item: Dict, criteria: str = "rag") -> Dict:
        """
        Оценка одного ответа
        
        Args:
            item: элемент из judge_evaluation_data_detailed.json
            criteria: критерии оценки
        """
        question = item.get('question', '')
        answer = item.get('generated_answer', '')
        contexts = item.get('contexts', [])
        
        if not answer:
            return {
                "error": "Empty answer",
                "total_score": 0,
                "verdict": "error"
            }
        
        prompt = self.build_judge_prompt(question, answer, contexts, criteria)
        
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096)
        
        if torch.cuda.is_available():
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=512,
                temperature=0.1,
                do_sample=False,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
                num_beams=1
            )
        
        response = self.tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        
        evaluation = self.parse_judge_response(response)
        evaluation['original_answer'] = answer
        evaluation['question'] = question
        
        return evaluation
    
    def evaluate_batch(self, items: List[Dict], criteria: str = "rag", 
                       output_file: str = None, save_interval: int = 10) -> List[Dict]:
        """
        Оценка батча ответов
        
        Args:
            items: список элементов для оценки
            criteria: критерии оценки
            output_file: файл для сохранения результатов
            save_interval: интервал сохранения
        """
        results = []
        
        print(f"\n{'='*60}")
        print(f"LLM-AS-A-JUDGE ОЦЕНКА")
        print(f"{'='*60}")
        print(f"Всего элементов: {len(items)}")
        print(f"Критерии: {criteria}")
        print(f"{'='*60}\n")
        
        start_time = time.time()
        
        for idx, item in enumerate(tqdm(items, desc="Оценка ответов")):
            try:
                evaluation = self.evaluate_single(item, criteria)
                evaluation['index'] = idx
                evaluation['timestamp'] = datetime.now().isoformat()
                results.append(evaluation)
                
                # Сохраняем промежуточные результаты
                if output_file and (idx + 1) % save_interval == 0:
                    self.save_results(results, output_file)
                    print(f"\n💾 Сохранено {len(results)} оценок")
                    
            except Exception as e:
                print(f"\n❌ Ошибка при оценке {idx}: {e}")
                results.append({
                    'index': idx,
                    'error': str(e),
                    'timestamp': datetime.now().isoformat()
                })
        
        elapsed = time.time() - start_time
        
        print(f"\n{'='*60}")
        print(f"ОЦЕНКА ЗАВЕРШЕНА")
        print(f"{'='*60}")
        print(f"✅ Успешно: {len([r for r in results if 'error' not in r])}")
        print(f"❌ Ошибок: {len([r for r in results if 'error' in r])}")
        print(f"⏱️  Время: {elapsed:.1f} сек")
        print(f"⚡ Скорость: {len(results)/elapsed:.2f} оценок/сек")
        
        return results
    
    def save_results(self, results: List[Dict], output_file: str):
        """Сохранение результатов"""
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        print(f"💾 Результаты сохранены в {output_file}")
    
    def generate_statistics(self, results: List[Dict]) -> pd.DataFrame:
        """
        Генерация статистики по оценкам
        """
        scores = []
        for r in results:
            if 'error' not in r:
                score_row = {}
                for key, value in r.items():
                    if isinstance(value, (int, float)) and key not in ['index', 'total_score']:
                        score_row[key] = value
                if 'total_score' in r:
                    score_row['total_score'] = r['total_score']
                if 'verdict' in r:
                    score_row['verdict'] = r['verdict']
                scores.append(score_row)
        
        df = pd.DataFrame(scores)
        
        if df.empty:
            print("⚠️ Нет данных для статистики")
            return df
        
        print(f"\n{'='*60}")
        print("СТАТИСТИКА ОЦЕНОК")
        print(f"{'='*60}")
        
        numeric_cols = [col for col in df.columns if col not in ['verdict']]
        
        for col in numeric_cols:
            if col in df.columns:
                print(f"\n📊 {col.upper()}:")
                print(f"   Среднее: {df[col].mean():.2f}")
                print(f"   Медиана: {df[col].median():.2f}")
                print(f"   Мин: {df[col].min():.2f}")
                print(f"   Макс: {df[col].max():.2f}")
                print(f"   Стд: {df[col].std():.2f}")
        
        if 'verdict' in df.columns:
            print(f"\n📋 РАСПРЕДЕЛЕНИЕ ВЕРДИКТОВ:")
            verdict_counts = df['verdict'].value_counts()
            for verdict, count in verdict_counts.items():
                print(f"   {verdict}: {count} ({count/len(df)*100:.1f}%)")
        
        return df


def main():
    
    # Файл с данными для оценки
    JUDGE_INPUT_FILE = "/kaggle/working/judge_evaluation_data_detailed.json"
    
    # Файлы для сохранения результатов
    JUDGE_OUTPUT_FILE = "llm_judge_results.json"
    STATISTICS_FILE = "evaluation_statistics.csv"
    
    # ============================================
    # МОДЕЛЬ ДЛЯ ОЦЕНКИ (открытые, без авторизации)
    # ============================================
    
    # Рекомендуемые модели-судьи:

    # 1. Qwen/Qwen3-4B (самая свежая и лучшая по метрика модель)
    #JUDGE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
    
    # 2. Qwen2.5-7B (лучшая для оценки, требует ~16GB VRAM)
    #JUDGE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
    
    # 3. Qwen2.5-3B (хороший баланс, ~6GB VRAM)
    JUDGE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
    
    # 4. Mistral-7B (мощная, ~16GB VRAM)
    # JUDGE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
    
    
    # Критерии оценки:
    # - "rag": для RAG-систем (релевантность, точность, полнота)
    # - "comprehensive": для детальных ответов (факты, ясность, инсайты)
    # - "summary": для кратких суммаризаций
    CRITERIA = "rag"
    

    USE_QUANTIZATION = True
    SAVE_INTERVAL = 5
    
    print("\n" + "="*60)
    print("🚀 LLM-AS-A-JUDGE ОЦЕНКА КАЧЕСТВА ОТВЕТОВ")
    print("="*60)
    print(f"📁 Входной файл: {JUDGE_INPUT_FILE}")
    print(f"🤖 Модель-судья: {JUDGE_MODEL}")
    print(f"⚙️  Критерии: {CRITERIA}")
    print(f"🔧 Квантизация: {USE_QUANTIZATION}")
    print("="*60 + "\n")
    
    print("Загрузка данных для оценки...")
    with open(JUDGE_INPUT_FILE, 'r', encoding='utf-8') as f:
        judge_data = json.load(f)
    
    print(f"Загружено {len(judge_data)} элементов для оценки")
    
    judge = LLMJudge(
        model_name=JUDGE_MODEL,
        use_quantization=USE_QUANTIZATION
    )
    
    results = judge.evaluate_batch(
        judge_data,
        criteria=CRITERIA,
        output_file=JUDGE_OUTPUT_FILE,
        save_interval=SAVE_INTERVAL
    )
    
    judge.save_results(results, JUDGE_OUTPUT_FILE)
    
    stats_df = judge.generate_statistics(results)
    
    if not stats_df.empty:
        stats_df.to_csv(STATISTICS_FILE, index=False, encoding='utf-8-sig')
        print(f"\n📊 Статистика сохранена в {STATISTICS_FILE}")
    
    print(f"\n{'='*60}")
    print("ПРИМЕРЫ ОЦЕНОК")
    print(f"{'='*60}")
    
    for i, result in enumerate(results[:3]):
        if 'error' not in result:
            print(f"\n📝 Пример {i+1}:")
            print(f"   Вердикт: {result.get('verdict', 'N/A')}")
            print(f"   Общий балл: {result.get('total_score', 0)}")
            print(f"   Сильные стороны: {result.get('strengths', 'N/A')[:100]}")
            print(f"   Слабые стороны: {result.get('weaknesses', 'N/A')[:100]}")
    
    print("\n" + "="*60)
    print("✅ ОЦЕНКА ЗАВЕРШЕНА!")
    print("="*60)
    print(f"📁 Результаты: {JUDGE_OUTPUT_FILE}")
    print(f"📊 Статистика: {STATISTICS_FILE}")
    print("="*60)


if __name__ == "__main__":
    main()


🚀 LLM-AS-A-JUDGE ОЦЕНКА КАЧЕСТВА ОТВЕТОВ
📁 Входной файл: /kaggle/working/judge_evaluation_data_detailed.json
🤖 Модель-судья: Qwen/Qwen2.5-3B-Instruct
⚙️  Критерии: rag
🔧 Квантизация: True

Загрузка данных для оценки...
Загружено 80 элементов для оценки
Загрузка модели-судьи Qwen/Qwen2.5-3B-Instruct на cuda...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

💾 Результаты сохранены в llm_judge_results.json

💾 Сохранено 70 оценок


Оценка ответов:  94%|█████████▍| 75/80 [35:40<02:22, 28.46s/it]

💾 Результаты сохранены в llm_judge_results.json

💾 Сохранено 75 оценок


Оценка ответов: 100%|██████████| 80/80 [38:02<00:00, 28.53s/it]

💾 Результаты сохранены в llm_judge_results.json

💾 Сохранено 80 оценок

ОЦЕНКА ЗАВЕРШЕНА
✅ Успешно: 77
❌ Ошибок: 3
⏱️  Время: 2282.6 сек
⚡ Скорость: 0.04 оценок/сек
💾 Результаты сохранены в llm_judge_results.json

СТАТИСТИКА ОЦЕНОК

📊 RELEVANCE:
   Среднее: 4.36
   Медиана: 5.00
   Мин: 2.00
   Макс: 5.00
   Стд: 0.81

📊 ACCURACY:
   Среднее: 4.22
   Медиана: 4.00
   Мин: 2.00
   Макс: 5.00
   Стд: 0.90

📊 COMPLETENESS:
   Среднее: 3.75
   Медиана: 4.00
   Мин: 2.00
   Макс: 5.00
   Стд: 1.09

📊 CONCISENESS:
   Среднее: 3.83
   Медиана: 4.00
   Мин: 2.00
   Макс: 5.00
   Стд: 0.75

📊 GROUNDEDNESS:
   Среднее: 4.25
   Медиана: 5.00
   Мин: 2.00
   Макс: 5.00
   Стд: 0.91

📊 TOTAL_SCORE:
   Среднее: 19.79
   Медиана: 19.00
   Мин: 10.00
   Макс: 25.00
   Стд: 4.26

📋 РАСПРЕДЕЛЕНИЕ ВЕРДИКТОВ:
   excellent: 34 (44.2%)
   good: 28 (36.4%)
   fair: 13 (16.9%)
   poor: 2 (2.6%)

📊 Статистика сохранена в evaluation_statistics.csv

ПРИМЕРЫ ОЦЕНОК

📝 Пример 1:
   Вердикт: fair
   Общий балл: 16
